# 22_rebalanced_dataset — decoy를 real_inactive에 맞춘 재균형 학습셋 (신규)

**한 줄 요약:** 쉬운 decoy가 학습을 지배하던 문제를 고치려고, **decoy 개수를 실측 inactive 개수만큼만** 남기고, **active는 전부 유지**한다.
**왜:** decoy가 너무 많아(1849개) 모델이 "합성물 감지기"가 되고 점수가 부풀려졌다(Chen 2019, PLOS ONE). decoy는 실측이 아니라 설계 변수라 줄여도 됨.
**구성:** active 2049(전부) + real_inactive 200(전부) + decoy 200(무작위 축소) = 2449.
**불균형:** active 2049 : inactive 400 (약 5:1) → 모델 노트북에서 `class_weight`로 보정(active를 안 버리는 방법).
**출력:** `canonical_smiles, source, potency, split`만 저장(특징은 v2 재사용).

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기
표 처리(pandas)와 층화 분할(train_test_split).

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())
import pandas as pd
from sklearn.model_selection import train_test_split

🔎 **코드 뜯어보기 (준비)**
- `from sklearn.model_selection import train_test_split` : 데이터를 층화(비율 유지) 분할하는 함수(20a에서 설명).

### 셀 1 — 재균형 구성
active 전부 + real_inactive 전부 + decoy를 real 개수만큼만 무작위로 골라 합친다.

In [ ]:
# 재균형 구성: active 전부 유지 + real_inactive 전부 + decoy를 real_inactive 개수만큼만
V2    = "data/HSD17B13_final_training_1to1_v2.csv"   # 특징표(지문+2D+3D+potency)
SRC   = "data/train_1to1.csv"                        # source(active/decoy/real_inactive) 라벨
OUT   = "data/HSD17B13_rebalanced_membership.csv"    # 어떤 분자가 어느 그룹/split인지만 저장

# v2에서 라벨만 읽고(가벼움) source를 붙임
base = pd.read_csv(V2, usecols=["canonical_smiles", "potency"])
src  = pd.read_csv(SRC)[["canonical_smiles", "source"]]
df   = base.merge(src, on="canonical_smiles", how="left")

act  = df[df.source == "active"]                 # 2049 전부 유지(active는 아까워서 안 버림)
real = df[df.source == "real_inactive"]          # 200 전부 유지(어려운 음성)
n_real = len(real)
# decoy는 real 개수만큼만 무작위로(random_state=42=재현). 쉬운 decoy가 지배하지 않도록
dec  = df[df.source == "decoy"].sample(n=n_real, random_state=42)

reb  = pd.concat([act, real, dec], ignore_index=True)
print("재균형 구성 →",
      "active", len(act), "| real_inactive", len(real), "| decoy(축소)", len(dec),
      "| 합계", len(reb))
print("  potency 분포:", dict(reb.potency.value_counts()),
      "→ active", len(act), ": inactive", len(real)+len(dec),
      f"(약 {len(act)/(len(real)+len(dec)):.1f}:1 불균형 → 모델에서 class_weight로 보정)")

🔎 **코드 뜯어보기 (셀 1)**
- `df[df.source == "active"]` : source 열이 'active'인 행만 고름(불리언 인덱싱).
- `df[df.source=="decoy"].sample(n=n_real, random_state=42)` : decoy 중 **real 개수만큼 무작위 추출**. `random_state=42`=매번 같은 표본(재현성).
- `pd.concat([...], ignore_index=True)` : 세 그룹을 위아래로 이어붙이고 인덱스 새로 매김.

### 셀 2 — source 층화 분할 & 저장
active/decoy/real_inactive 세 그룹이 train/val/test에 비율대로 들어가게 나눈다(특히 test에 real_inactive가 꼭 포함되도록).

In [ ]:
# source로 '층화 분할' → train/val/test 각각에 active·decoy·real_inactive가 비율대로 들어감
TEST, VAL = 0.15, 0.15
strat = reb["source"]                            # 3그룹(active/decoy/real) 비율 유지가 목적
tr_val, te = train_test_split(reb, test_size=TEST, stratify=strat, random_state=42)
val_ratio  = VAL / (1 - TEST)
tr, va = train_test_split(tr_val, test_size=val_ratio, stratify=tr_val["source"], random_state=42)

reb2 = pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")])
reb2 = reb2[["canonical_smiles", "source", "potency", "split"]]
reb2.to_csv(OUT, index=False)

print("분할 결과(그룹별):")
print(pd.crosstab(reb2["split"], reb2["source"]).to_string())
print("\n저장:", OUT, "| 총", len(reb2), "행")
print("→ 모델 노트북(23)이 이 파일로 어떤 분자를 어느 split에 쓸지 결정하고, 특징은 v2에서 가져옴")

🔎 **코드 뜯어보기 (셀 2)**
- `stratify=reb["source"]` : potency(0/1) 대신 **3그룹**으로 층화 → test에도 어려운 real_inactive가 비율대로 들어가 정직한 평가 가능.
- `val_ratio = VAL/(1-TEST)` : 남은 85% 중 몇 %를 떼야 전체 15%가 되는지 환산(20a와 동일).
- `pd.crosstab(split, source)` : split×source 교차표로 분할이 잘 됐는지 확인.